# Strings and Arrays

## Quick notes

### What this covers
- Parameter expansion in full: length, substrings, removing prefixes/suffixes, replacing, defaults.
- Case conversion, trimming, splitting and joining, with portable tools for bash 3.2.
- Indexed arrays (create, append, slice, count, sparse), `"${a[@]}"` vs `"${a[*]}"`, reading lines into arrays.
- Associative arrays: why they are missing on this machine and what to do instead.
- String and number comparison, `printf` formatting in depth, here-strings, `tr`, reversing and repeating.

### Key points
- Parameter expansion (`${var...}`) does most string work with no external process, so it is fast and safe.
- `#`/`##` strip from the front, `%`/`%%` strip from the back (single = shortest match, double = longest). Patterns are globs, not regular expressions.
- Arrays: `a=(x y z)`, `${a[1]}`, `"${a[@]}"` (all, one word each), `${#a[@]}` (count), `${!a[@]}` (indices).
- Always quote expansions: `"$var"`, `"${a[@]}"`.

### Gotchas (the part worth revising twice)
- `${a}` of an array is only element 0. `$a[1]` is `${a}` followed by the text `[1]`.
- Unquoted `${a[@]}` or `$var` is split on whitespace and globbed. Elements with spaces break.
- `${s: -3}` needs the space (or parentheses), otherwise it is read as the `:-` default operator.
- `[ "$a" \< "$b" ]` and `[[ $a < $b ]]` compare as strings. Use `-lt` or `(( ))` for numbers. Numbers with a leading zero (`08`) are octal inside `(( ))`.
- `declare -A`, `${s^^}` and `mapfile` need bash 4+; negative array subscripts need bash 4.3. This machine has bash 3.2: they are shown as non-run blocks or version-guarded cells.
- `tr`, `sed`, `awk`, `printf` differ a little between GNU (Linux) and BSD (macOS); the demos use only the common core.

### When it matters
- Parsing file names, paths, CSV-ish lines, version numbers, config values.
- Any script that handles lists of things (files, hosts, arguments).

### One-line recap example
```bash
f=/data/logs/app.tar.gz; echo "${f##*/} ${f%%.*} ${f#*.}"    # app.tar.gz /data/logs/app tar.gz
```

Related: `01-Basics/01-Fundamentals` (quoting and expansion order), `01-Basics/02-Control-Flow` (tests and loops), `01-Basics/03-Functions-And-Arguments` (passing arrays to functions), `02-Streams-And-Text/02-Text-Processing-Tools` (sed/awk for bigger jobs).

## 01. Length and substrings
- `${#s}` is the length in characters (bytes under `LC_ALL=C`, which these notebooks use).
- `${s:offset}` from offset to the end; `${s:offset:length}` a slice. Offsets start at 0.
- A negative offset counts from the end, but it needs a space or parentheses: `${s: -3}` or `${s:(-3)}`.
- Without the space, `${s:-3}` means "use 3 if `s` is empty": a classic trap.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
s="Hello, World"
echo "length: ${#s}"
echo "from 7:   [${s:7}]"
echo "7 for 3:  [${s:7:3}]"
echo "last 5:   [${s: -5}]"
echo "last 5:   [${s:(-5)}]"
echo "trap: [${s:-5}]   (that is the default-value operator, s is not empty so it prints s)"

length: 12
from 7:   [World]
7 for 3:  [Wor]
last 5:   [World]
last 5:   [World]
trap: [Hello, World]   (that is the default-value operator, s is not empty so it prints s)


## 02. Removing patterns: `#` `##` `%` `%%`
- Remember by keyboard: `#` is on the left of `$` (front), `%` is on the right (back).
- One symbol removes the *shortest* match, two remove the *longest*.
- The pattern is a glob (`*`, `?`, `[...]`), matched against the whole front or back of the value.
- Nothing matches, nothing is removed.

In [3]:
%%bash
f=/data/logs/app.tar.gz
echo "# */   : ${f#*/}"
echo "## */  : ${f##*/}"
echo "% .*   : ${f%.*}"
echo "%% .*  : ${f%%.*}"
echo "# .*   : ${f#.*}   (the value does not start with a dot, so unchanged)"
echo "ext    : ${f##*.}"
x=${f##*/}; echo "stem   : ${x%%.*}"

# */   : data/logs/app.tar.gz
## */  : app.tar.gz
% .*   : /data/logs/app.tar
%% .*  : /data/logs/app
# .*   : /data/logs/app.tar.gz   (the value does not start with a dot, so unchanged)
ext    : gz
stem   : app


## 03. `basename` and `dirname` without a subprocess
- `${path##*/}` is `basename`, `${path%/*}` is `dirname`. Both are pure bash, no fork.
- Edge cases differ: no slash in the path, trailing slash, root. The real tools handle these; the emulation needs care.
- Use the real `basename`/`dirname` when correctness on odd input matters, and the expansions in tight loops.

In [4]:
%%bash
for p in /usr/lib/libz.so a/b/c.txt file.txt /top dir/; do
  printf '%-14s  expansion: base=[%s] dir=[%s]   real: base=[%s] dir=[%s]\n' \
    "$p" "${p##*/}" "${p%/*}" "$(basename "$p")" "$(dirname "$p")"
done

/usr/lib/libz.so  expansion: base=[libz.so] dir=[/usr/lib]   real: base=[libz.so] dir=[/usr/lib]
a/b/c.txt       expansion: base=[c.txt] dir=[a/b]   real: base=[c.txt] dir=[a/b]
file.txt        expansion: base=[file.txt] dir=[file.txt]   real: base=[file.txt] dir=[.]
/top            expansion: base=[top] dir=[]   real: base=[top] dir=[/]
dir/            expansion: base=[] dir=[dir]   real: base=[dir] dir=[.]


## 04. Replacing text: `/` `//` `/#` `/%`
- `${s/pat/rep}` replaces the first match, `${s//pat/rep}` all matches.
- `${s/#pat/rep}` only at the start, `${s/%pat/rep}` only at the end.
- Leave out the replacement to delete: `${s//pat}`.
- The pattern is a glob; `*` and `?` work, regex does not. Quote the pattern part to make characters literal.

In [5]:
%%bash
s="the cat sat on the mat"
echo "first:  ${s/at/AT}"
echo "all:    ${s//at/AT}"
echo "start:  ${s/#the/A}"
echo "end:    ${s/%mat/rug}"
echo "delete: ${s//a}"
echo "class:  ${s//[aeiou]/_}"
echo "glob:   ${s/c*t/<cut>}   (greedy: from the first c to the LAST t)"
p="a.b.c"
echo "dots:   ${p//./-}"
path="a/b/c"
echo "slash:  ${path//\//:}"

first:  the cAT sat on the mat
all:    the cAT sAT on the mAT
start:  A cat sat on the mat
end:    the cat sat on the rug
delete: the ct st on the mt
class:  th_ c_t s_t _n th_ m_t
glob:   the <cut>   (greedy: from the first c to the LAST t)
dots:   a-b-c
slash:  a:b:c


## 05. Patterns are globs, not regular expressions
- In `${...}` and in `case` or `[[ x == pat ]]`, the pattern language is the same glob language as file names.
- `*` any string, `?` one char, `[abc]`, `[a-z]`, `[!x]` not x.
- Quoting disables the special meaning: `[[ $s == "a*" ]]` compares literally.
- For real regex use `[[ $s =~ regex ]]` (works in 3.2; the match goes in `BASH_REMATCH`) or `sed`/`awk`/`grep`.

In [6]:
%%bash
v="release-2024-05.txt"
case $v in
  release-*.txt) echo "case glob matched" ;;
esac
[[ $v == release-????-??.txt ]] && echo "? matched single characters"
[[ $v == "release-*" ]] || echo "quoted * is literal, no match"
if [[ $v =~ ^release-([0-9]{4})-([0-9]{2})\.txt$ ]]; then
  echo "year=${BASH_REMATCH[1]} month=${BASH_REMATCH[2]}"
fi

case glob matched
? matched single characters
quoted * is literal, no match
year=2024 month=05


## 06. Case conversion
- bash 4+ has `${s^^}` (upper), `${s,,}` (lower), `${s^}` (first letter upper). Not available on 3.2.
- Portable: `tr '[:lower:]' '[:upper:]'` for whole strings. Use character classes, not `a-z`: they are correct in every locale.
- Capitalise only the first letter on 3.2: split the first character with `${s:0:1}` and convert just that.
- `awk '{print toupper($0)}'` works too. `sed 's/.*/\U&/'` is GNU only: BSD `sed` on macOS does not support `\U`.

In [7]:
%%bash
s="hello World"
echo "upper: $(printf '%s' "$s" | tr '[:lower:]' '[:upper:]')"
echo "lower: $(printf '%s' "$s" | tr '[:upper:]' '[:lower:]')"
echo "awk:   $(echo "$s" | awk '{print toupper($0)}')"
first=$(printf '%s' "${s:0:1}" | tr '[:lower:]' '[:upper:]')
echo "capitalised: $first${s:1}"
to_lower() { printf '%s' "$1" | tr '[:upper:]' '[:lower:]'; }
[ "$(to_lower "ABC")" = "$(to_lower "abc")" ] && echo "case-insensitive equal"

upper: HELLO WORLD
lower: hello world
awk:   HELLO WORLD
capitalised: Hello World
case-insensitive equal


In [8]:
%%bash
# needs bash 4+ (not run here): version-guarded so the cell is safe on 3.2
if (( BASH_VERSINFO[0] >= 4 )); then
  eval 's="hello World"; echo "${s^^} ${s,,} ${s^}"'
else
  echo "skipped: needs bash 4+ (this is bash ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]})"
fi

skipped: needs bash 4+ (this is bash 3.2)


## 07. Trimming whitespace
- There is no built-in trim. Choices: pure-bash with parameter expansion, `sed`, `awk`, or `read`.
- Pure bash: remove the longest run of leading spaces with `##`, trailing with `%%`, using a character class.
- Trick: `read -r x <<< "$s"` trims leading and trailing blanks (the default `IFS`) and keeps inner spaces.
- `echo $s` (unquoted) also collapses inner runs of whitespace, usually not wanted.

In [9]:
%%bash
s=$'   \t  padded  text \t '
trim() {                                    # prints the trimmed value
  local v=$1
  v=${v#"${v%%[![:space:]]*}"}              # strip leading whitespace
  v=${v%"${v##*[![:space:]]}"}              # strip trailing whitespace
  printf '%s' "$v"
}
echo "[$(trim "$s")]"
read -r viaread <<< "$s"; echo "[$viaread]"
echo "sed: [$(printf '%s' "$s" | sed 's/^[[:space:]]*//; s/[[:space:]]*$//')]"
echo "unquoted echo collapses inner runs: [$(echo $s)]"

[padded  text]
[padded  text]
sed: [padded  text]
unquoted echo collapses inner runs: [padded text]


## 08. Splitting a string
- Set `IFS` for one command to choose the separator, then `read -ra arr <<< "$s"` splits into an array.
- `IFS=, read ...` changes `IFS` only for that `read` (a temporary assignment), so nothing leaks.
- Whitespace separators collapse runs (`a   b` is two fields); non-whitespace separators do not (`a,,b` has an empty middle field).
- `read` only reads one line, so a value containing a newline is cut at that point.
- `-r` stops `read` from treating backslashes as escapes. Always use it.

In [10]:
%%bash
csv="alice,30,,paris"
IFS=, read -ra f <<< "$csv"
echo "${#f[@]} fields:"
for i in "${!f[@]}"; do echo "  [$i]=[${f[$i]}]"; done
IFS=: read -ra parts <<< "/usr/bin:/bin:/sbin"
printf 'path part: %s\n' "${parts[@]}"
IFS=' ' read -ra w <<< "  many    spaces   here "
echo "${#w[@]} words: ${w[*]}"
read -r a b rest <<< "one two three four"      # last variable gets the rest
echo "a=$a b=$b rest=[$rest]"
echo "IFS unchanged after the temporary assignment: $(printf '%s' "$IFS" | od -An -c | tr -s ' ')"

4 fields:
  [0]=[alice]
  [1]=[30]
  [2]=[]
  [3]=[paris]
path part: /usr/bin
path part: /bin
path part: /sbin
3 words: many spaces here
a=one b=two rest=[three four]
IFS unchanged after the temporary assignment:  \t \n 


In [11]:
%%bash
# splitting with other tools: tr turns a separator into newlines, cut picks a field
echo "a:b:c" | tr ':' '\n'
echo "a:b:c" | cut -d: -f2
# parameter expansion walk: peel one field at a time
rest="x/y/z"
while [ -n "$rest" ]; do
  case $rest in
    */*) head=${rest%%/*}; rest=${rest#*/} ;;
    *)   head=$rest; rest= ;;
  esac
  echo "field: $head"
done

a
b
c
b
field: x
field: y
field: z


## 09. Joining
- `"${a[*]}"` joins array elements with the first character of `IFS` (a space by default).
- Set `IFS` in a subshell or function (`local IFS=,`) so you do not change the caller's `IFS`.
- Only the first character of `IFS` is used, so a multi-character separator needs a loop or `sed`/`paste`.
- `printf '%s, ' "${a[@]}"` leaves a trailing separator; trim it with `${s%, }`.

In [12]:
%%bash
a=(red green blue)
echo "default: ${a[*]}"
join() { local IFS=$1; shift; echo "$*"; }
join , "${a[@]}"
join '; ' "${a[@]}"            # only ';' is used
s=$(printf '%s, ' "${a[@]}"); echo "[${s%, }]"
printf '%s\n' "${a[@]}" | paste -sd'|' -
printf '%s\n' "${a[@]}" | tr '\n' ' '; echo

default: red green blue
red,green,blue
red;green;blue
[red, green, blue]
red|green|blue
red green blue 


## 10. Indexed arrays: create, read, append, count
- Create with `a=(x y z)`, or one element at a time: `a[0]=x`.
- Read one element `${a[1]}`; all elements `"${a[@]}"`; the count `${#a[@]}`; the length of one element `${#a[1]}`.
- Append with `a+=(more)` (bash 3.1+). `a+=x` (no parentheses) appends text to element 0, which is rarely what you want.
- Plain `$a` or `${a}` is just element 0.
- An element may hold spaces if it is quoted at creation.

In [13]:
%%bash
fruits=(apple "dragon fruit" cherry)
echo "second: ${fruits[1]}"
echo "count: ${#fruits[@]}   length of element 1: ${#fruits[1]}"
fruits+=(date elderberry)
echo "after append: ${#fruits[@]} items, last: ${fruits[${#fruits[@]}-1]}"
echo "plain \$fruits is element 0: $fruits"
fruits+=!                       # no parentheses: glues text onto element 0
echo "element 0 now: ${fruits[0]}"
empty=(); echo "empty array count: ${#empty[@]}"

second: dragon fruit
count: 3   length of element 1: 12
after append: 5 items, last: elderberry
plain $fruits is element 0: apple
element 0 now: apple!
empty array count: 0


## 11. Slices, indices, `unset`, and sparse arrays
- Slice: `"${a[@]:start:count}"`. Without `count`, up to the end.
- Indices: `"${!a[@]}"` lists the subscripts that exist. Use it to loop safely over sparse arrays.
- `unset 'a[1]'` removes one element but does NOT renumber the rest: you get a hole (a sparse array).
- `${#a[@]}` counts the elements that exist, not the highest index.
- To compact an array, rebuild it: `a=("${a[@]}")`.
- Negative subscripts (`${a[-1]}`) need bash 4.3. On 3.2 use `${a[${#a[@]}-1]}` for the last element (works for non-sparse arrays).

In [14]:
%%bash
a=(zero one two three four)
echo "slice 1,2: ${a[@]:1:2}"
echo "slice 3..: ${a[@]:3}"
unset 'a[1]'
echo "count=${#a[@]} indices=(${!a[@]})"
echo "values: ${a[@]}"
echo "a[1] is now [${a[1]-<unset>}]"
a[10]=ten
echo "after a[10]: count=${#a[@]} indices=(${!a[@]})"
a=("${a[@]}")                   # compact
echo "compacted: indices=(${!a[@]})"
echo "last: ${a[${#a[@]}-1]}"

slice 1,2: one two
slice 3..: three four
count=4 indices=(0 2 3 4)
values: zero two three four
a[1] is now [<unset>]
after a[10]: count=5 indices=(0 2 3 4 10)
compacted: indices=(0 1 2 3 4)
last: ten


In [15]:
%%bash expect_fail
# negative subscript on bash 3.2: the index is evaluated as arithmetic and -1 is a bad subscript
b=(x y z)
echo "${b[-1]}"
echo "after: still running"

script: line 3: b: bad array subscript

after: still running


## 12. `"${a[@]}"` vs `"${a[*]}"` vs unquoted
- `"${a[@]}"` expands to one word per element, spaces inside elements preserved. This is the one to use for passing and looping.
- `"${a[*]}"` is a single word (elements joined with the first character of `IFS`).
- Unquoted `${a[@]}` or `${a[*]}` split every element on whitespace and glob the pieces.
- Empty elements: `"${a[@]}"` keeps them as empty words; unquoted drops them.

In [16]:
%%bash
args() { printf 'count=%d:' "$#"; for x in "$@"; do printf ' <%s>' "$x"; done; echo; }
a=("two words" "" "x*" plain)
echo '"${a[@]}"'; args "${a[@]}"
echo '"${a[*]}"'; args "${a[*]}"
echo '${a[@]}   (unquoted)'; args ${a[@]}

"${a[@]}"
count=4: <two words> <> <x*> <plain>
"${a[*]}"
count=1: <two words  x* plain>
${a[@]}   (unquoted)
count=4: <two> <words> <x*> <plain>


In [17]:
%%bash fresh expect_fail
# with set -u, bash 3.2 treats an EMPTY array as unset when expanded: "unbound variable"
set -u
e=()
echo "count is fine: ${#e[@]}"
echo "expand: [${e[@]}]"
echo "never reached"

count is fine: 0
script: line 5: e[@]: unbound variable
[exit status 1]


In [18]:
%%bash
# portable guard that works under set -u on any version: ${e[@]+"${e[@]}"}
( set -u; e=(); args() { echo "got $# args"; }; args ${e[@]+"${e[@]}"}; e=(a b); args ${e[@]+"${e[@]}"} )

got 0 args
got 2 args


## 13. Looping and copying arrays
- `for x in "${a[@]}"` visits the elements. `for i in "${!a[@]}"` visits indices (use when you need the position or the array is sparse).
- C-style loop `for ((i=0; i<${#a[@]}; i++))` works for dense arrays.
- Copy with `b=("${a[@]}")`. `b=$a` copies only element 0.
- Copying a sparse array this way renumbers it from 0.
- Modify while iterating by index, not by value.

In [19]:
%%bash
a=("a b" c d)
b=("${a[@]}")
b[0]="CHANGED"
echo "a: ${a[*]}   b: ${b[*]}"
bad=$a; echo "bad copy is a plain string: [$bad]"
for i in "${!a[@]}"; do echo "$i -> ${a[$i]}"; done
nums=(1 2 3 4)
for ((i = 0; i < ${#nums[@]}; i++)); do nums[$i]=$(( nums[i] * nums[i] )); done
echo "squares: ${nums[*]}"
# apply an operation to every element at once
files=(a.txt b.txt c.log)
echo "${files[@]/.txt/.bak}"
echo "${files[@]/%.txt/.bak}"
echo "lengths: $(for f in "${files[@]}"; do printf '%d ' "${#f}"; done)"

a: a b c d   b: CHANGED c d
bad copy is a plain string: [a b]
0 -> a b
1 -> c
2 -> d
squares: 1 4 9 16
a.bak b.bak c.log
a.bak b.bak c.log
lengths: 5 5 5 


## 14. Arrays and word splitting: build commands safely
- Use an array to hold a command's arguments when some contain spaces; run it with `"${cmd[@]}"`.
- A plain string such as `opts="-a 'b c'"` cannot be reliably re-split: the quotes are just characters.
- Never `eval` user data to rebuild a command; arrays remove the need.
- Arrays also give conditional arguments: `extra=(); [ -n "$v" ] && extra+=(--verbose)`.

In [20]:
%%bash
args() { printf 'count=%d:' "$#"; for x in "$@"; do printf ' <%s>' "$x"; done; echo; }
opts="-a 'b c'"
echo "string version:"; args $opts
cmd=(-a "b c")
echo "array version:";  args "${cmd[@]}"
verbose=1; extra=()
[ -n "$verbose" ] && extra+=(--verbose)
args base "${extra[@]}" end

string version:
count=3: <-a> <'b> <c'>
array version:
count=2: <-a> <b c>
count=3: <base> <--verbose> <end>


## 15. Reading lines into an array (3.2 way, and `mapfile`)
- bash 3.2 has no `mapfile`/`readarray`. Use a `while IFS= read -r` loop and `+=`.
- Feed the loop from a file (`< file`) or from process substitution (`< <(cmd)`), not from a pipe, or the array is built in a subshell and lost.
- `IFS=` keeps leading/trailing spaces; `-r` keeps backslashes.
- A last line with no trailing newline is skipped by `read` unless you also test `|| [ -n "$line" ]`.
- Globbing straight into an array (`logs=(*.log)`) is the safe way to hold file names (if nothing matches, you get the literal pattern).

In [21]:
%%bash
printf 'first line\n  indented line\nthird\\n has backslash\n' > lines.txt
lines=()
while IFS= read -r line; do lines+=("$line"); done < lines.txt
echo "read ${#lines[@]} lines"
for i in "${!lines[@]}"; do echo "[$i] <${lines[$i]}>"; done
# the pipeline trap: the loop runs in a subshell
piped=()
cat lines.txt | while IFS= read -r line; do piped+=("$line"); done
echo "array filled through a pipe has ${#piped[@]} items"
touch b.log a.log c.log
logs=(*.log)
echo "logs: ${logs[*]} (count ${#logs[@]})"
# last line without a newline
printf 'x\ny' > nonl.txt
got=(); while IFS= read -r l || [ -n "$l" ]; do got+=("$l"); done < nonl.txt
echo "with the -n guard: ${#got[@]} lines (${got[*]})"
short=(); while IFS= read -r l; do short+=("$l"); done < nonl.txt
echo "without the guard: ${#short[@]} lines (${short[*]})"
rm -f lines.txt nonl.txt *.log

read 3 lines
[0] <first line>
[1] <  indented line>
[2] <third\n has backslash>
array filled through a pipe has 0 items
logs: a.log b.log c.log (count 3)
with the -n guard: 2 lines (x y)
without the guard: 1 lines (x)


In [22]:
%%bash
# needs bash 4+ (not run here): mapfile reads all lines in one command
if (( BASH_VERSINFO[0] >= 4 )); then
  eval 'mapfile -t arr < <(printf "a\nb\nc\n"); echo "${#arr[@]} lines: ${arr[*]}"'
else
  echo "skipped: mapfile/readarray need bash 4+ (this is bash ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]})"
fi

skipped: mapfile/readarray need bash 4+ (this is bash 3.2)


## 16. Associative arrays (key to value): not on bash 3.2
- `declare -A m; m[key]=value` needs bash 4+. On bash 3.2 `declare -A` is an error, and the code that follows keeps running with silently wrong results (shown below). Guard with `(( BASH_VERSINFO[0] >= 4 ))` when a script needs it.
- Workaround 1: **parallel arrays**: `keys=(a b)` and `vals=(1 2)` with a lookup function. Simple, fine for small tables.
- Workaround 2: **key=value lines** in a variable or file, queried with `awk`. Good for lookups from data.
- Workaround 3: **dynamic variable names**: store `map_KEY=value` with `printf -v`, read with `${!name}`. Keys must be valid identifier characters, so sanitise them (and note that two different keys can end up with the same name).
- Pick one and wrap it in `map_set`/`map_get` functions so you can swap in `declare -A` on bash 4+.

In [23]:
%%bash expect_fail
declare -A age
age[alice]=30
age[bob]=25
# On bash 3.2 the script does not stop after the failed declare. Without it, age is a plain INDEXED array and the
# subscripts "alice" and "bob" are evaluated as arithmetic (unset variables = 0), so both land on index 0.
echo "indices: (${!age[@]})  count: ${#age[@]}  value at every key: ${age[alice]} ${age[bob]}"

script: line 1: declare: -A: invalid option
declare: usage: declare [-afFirtx] [-p] [name[=value] ...]
indices: (0)  count: 1  value at every key: 25 25


In [24]:
%%bash
# workaround 1: parallel arrays + lookup
keys=(alice bob carol); vals=(30 25 41)
lookup() {                         # lookup KEY -> prints value, status 1 if missing
  local i
  for i in "${!keys[@]}"; do
    if [ "${keys[$i]}" = "$1" ]; then echo "${vals[$i]}"; return 0; fi
  done
  return 1
}
echo "bob=$(lookup bob)"
lookup dave || echo "dave: not found (status $?)"

bob=25
dave: not found (status 1)


In [25]:
%%bash
# workaround 2: key=value lines queried with awk
table=$'alice=30\nbob=25\ncarol=41'
get() { printf '%s\n' "$table" | awk -F= -v k="$1" '$1 == k { print substr($0, length(k) + 2); found = 1 } END { exit !found }'; }
get carol
get zed || echo "zed: not found"
# update: rebuild without the key, then add it back
set_kv() { table=$(printf '%s\n' "$table" | awk -F= -v k="$1" '$1 != k'; printf '%s=%s\n' "$1" "$2"); }
set_kv bob 26
printf '%s\n' "$table" | sort

41
zed: not found
alice=30
bob=26
carol=41


In [26]:
%%bash
# workaround 3: dynamic variable names (map_<key>), sanitised keys
map_set() { local k=${1//[^A-Za-z0-9_]/_}; printf -v "map_$k" '%s' "$2"; }
map_get() { local k=${1//[^A-Za-z0-9_]/_}; local v="map_$k"; printf '%s\n' "${!v}"; }
map_set alice 30
map_set "web-server" "10.0.0.5"
map_get alice
map_get web-server
echo "all map variables: $(set | grep '^map_[A-Za-z0-9_]*=' | cut -d= -f1 | sort | tr '\n' ' ')"

30
10.0.0.5
all map variables: map_alice map_web_server 


The bash 4+ version (not run here, since this machine only has bash 3.2):

```bash
# needs bash 4+ (not run here)
declare -A age=([alice]=30 [bob]=25)
age[carol]=41
echo "${age[alice]}"              # 30
for k in "${!age[@]}"; do echo "$k=${age[$k]}"; done   # order is NOT guaranteed
echo "${#age[@]}"                 # 3
[[ -v age[bob] ]] && echo "bob present"                # -v on elements needs bash 4.3
unset 'age[bob]'
```

## 17. String comparison
- Inside `[[ ... ]]` use `==` / `!=` (pattern match on the right side unless quoted), `<` / `>` (string order), `=~` (regex).
- In `[ ... ]` use `=` and `!=`; `<` and `>` must be escaped (`\<`) or they become redirections.
- `-z` is true for an empty string, `-n` for a non-empty one. Always quote variables inside `[ ]`.
- Ordering uses the locale's collation; with `LC_ALL=C` it is plain byte order (uppercase before lowercase). Do not rely on it across machines.
- Case-insensitive compare on bash 3.2: lowercase both sides with `tr`, or use `shopt -s nocasematch` (works for `[[ ]]` and `case`).

In [27]:
%%bash
a="apple"; b="Banana"
[ "$a" = "apple" ] && echo "= matches"
[[ $a == a* ]] && echo "pattern a* matches"
[[ $a == "a*" ]] || echo "quoted pattern is literal"
if [[ $a < $b ]]; then echo "apple < Banana"; else echo "apple > Banana (byte order in the C locale: uppercase sorts first)"; fi
[ -z "" ] && echo "-z: empty"
[ -n "$a" ] && echo "-n: not empty"
shopt -s nocasematch
[[ HELLO == hello ]] && echo "nocasematch: equal"
shopt -u nocasematch
[[ HELLO == hello ]] || echo "back to case-sensitive"
# the unquoted-variable trap: an empty variable vanishes and the test is malformed
empty=""
[ $empty = "x" ] 2>&1 || echo "unquoted empty variable: test breaks (status $?)"
[ "$empty" = "x" ] || echo "quoted: fine, just false"

= matches
pattern a* matches
quoted pattern is literal
apple > Banana (byte order in the C locale: uppercase sorts first)
-z: empty
-n: not empty
nocasematch: equal
back to case-sensitive
script: line 14: [: =: unary operator expected
unquoted empty variable: test breaks (status 2)
quoted: fine, just false


## 18. Numeric strings
- Variables are strings; arithmetic (`$(( ))`, `(( ))`, `[ -lt ]`) interprets them as numbers.
- Compare numbers with `-eq -ne -lt -le -gt -ge` or `(( a < b ))`. `=` compares text: `"01" = "1"` is false.
- A leading zero means octal in arithmetic: `08` and `09` are errors, `010` is 8. Force decimal with `10#08`.
- Check "is an integer" with a `case` pattern or `[[ =~ ]]` before doing math on user input.
- Bash does integer math only. For decimals use `awk` or `bc` (if installed).

In [28]:
%%bash
[ "01" = "1" ] || echo '"01" = "1" is false (string compare)'
[ "01" -eq 1 ] && echo '"01" -eq 1 is true (numeric compare)'
echo "010 in arithmetic: $(( 010 ))   (octal!)"
echo "10#010: $(( 10#010 ))"
x=08
( echo $(( x + 1 )) ) 2>&1 | head -1
echo "forced decimal: $(( 10#$x + 1 ))"
is_int() { case $1 in ''|*[!0-9]*) return 1;; esac; }
for v in 42 4x2 "" 007; do if is_int "$v"; then echo "[$v] integer"; else echo "[$v] not an integer"; fi; done
echo "integer division: $(( 7 / 2 )), remainder $(( 7 % 2 ))"
echo "decimals via awk: $(awk 'BEGIN { printf "%.2f", 7 / 2 }')"

"01" = "1" is false (string compare)
"01" -eq 1 is true (numeric compare)
010 in arithmetic: 8   (octal!)
10#010: 10
script: line 6: 08: value too great for base (error token is "08")
forced decimal: 9
[42] integer
[4x2] not an integer
[] not an integer
[007] integer
integer division: 3, remainder 1
decimals via awk: 3.50


## 19. `printf` in depth
- `printf FORMAT args...` is the portable, predictable way to print. Prefer it over `echo` for anything with variables.
- Common conversions: `%s` string, `%d` integer, `%f` float, `%x`/`%X` hex, `%o` octal, `%c` first character, `%%` a percent sign, `%b` expand backslash escapes in the argument, `%q` shell-quoted output (re-usable as input).
- Width and flags: `%5d` right-aligned in 5, `%-10s` left-aligned in 10, `%05d` zero padded, `%+d` always show sign, `%.2f` two decimals, `%8.3f`, `%.3s` truncate to 3 chars, `%*d` width from an argument.
- The format is reused until all arguments are consumed. Missing arguments become empty/0.
- `printf -v var` stores the result in a variable instead of printing (bash 3.1+).
- Format strings starting with `-` need `--` (`printf -- "-%s\n" x`).

In [29]:
%%bash
printf '[%s] [%5s] [%-5s] [%.2s]\n' abc abc abc abc
printf '[%d] [%5d] [%-5d] [%05d] [%+d]\n' 42 42 42 42 42
printf '[%f] [%.2f] [%8.3f] [%e]\n' 3.14159 3.14159 3.14159 31415.9
printf 'hex %x  HEX %X  octal %o  char %c  percent %%\n' 255 255 8 hello
printf 'dynamic width: [%*d] [%-*d]\n' 6 7 6 7
printf '%s and %s\n' a b c d           # format reused: 4 arguments, 2 per pass
printf '%d items\n'                     # missing argument becomes 0
printf '%b' 'tab:\there\nnew line\n'
printf '%s' 'literal \t is kept by %s'; echo
printf -v out '%03d-%s' 7 x; echo "$out"

[abc] [  abc] [abc  ] [ab]
[42] [   42] [42   ] [00042] [+42]
[3.141590] [3.14] [   3.142] [3.141590e+04]
hex ff  HEX FF  octal 10  char h  percent %
dynamic width: [     7] [7     ]
a and b
c and d
0 items
tab:	here
new line
literal \t is kept by %s
007-x


In [30]:
%%bash
# %q quotes a value so the shell reads it back identically
tricky="it's a \"test\" with spaces & \$vars"
printf '%q\n' "$tricky"
q=$(printf '%q' "$tricky")
eval "back=$q"   # only safe because %q escaped the value first
[ "$back" = "$tricky" ] && echo "round trip OK"
# a table with columns
printf '%-8s %6s %6s\n' name qty price
printf '%-8s %6d %6.2f\n' apple 3 0.5 pear 12 1.25 melon 1 3
# echo vs printf: echo options and escapes vary between shells
echo -n "no newline with -n"; echo
printf -- '-n is just text here: %s\n' "-n"

it\'s\ a\ \"test\"\ with\ spaces\ \&\ \$vars
round trip OK
name        qty  price
apple         3   0.50
pear         12   1.25
melon         1   3.00
no newline with -n
-n is just text here: -n


## 20. Here-strings, here-docs and `tr`
- `cmd <<< "text"` feeds `text` plus a newline to `cmd`'s stdin. It avoids `echo "$x" | cmd` and keeps loops in the current shell.
- Here-docs (`<<DELIM`) are for multi-line input. Quote the delimiter (`<<'DELIM'`) to stop expansion.
- `tr` translates or deletes single characters: `tr a-z A-Z`, `tr -d chars`, `tr -s chars` (squeeze repeats), `tr -c` (complement).
- `tr` works on characters, not words, and takes no regex. For words use `sed`.
- `tr` reads stdin only, never file names.

In [31]:
%%bash
read -r first second <<< "alpha beta"; echo "$second $first"
echo "remove digits:  $(tr -d '0-9' <<< "a1b22c333")"
echo "squeeze spaces: $(tr -s ' ' <<< "a    b     c")"
echo "keep letters:   $(tr -cd '[:alpha:]' <<< "a1-b2_c3")"
echo "rot13:          $(tr 'A-Za-z' 'N-ZA-Mn-za-m' <<< "Hello")"
echo "count lines:    $(wc -l <<< $'a\nb\nc' | tr -d ' ')"
cat <<'DOC'
single-quoted delimiter: $HOME and $(date) stay literal
DOC
name=Ada
cat <<DOC
unquoted delimiter: hello $name, 2+3=$((2+3))
DOC

beta alpha
remove digits:  abc
squeeze spaces: a b c
keep letters:   abc
rot13:          Uryyb
count lines:    3
single-quoted delimiter: $HOME and $(date) stay literal
unquoted delimiter: hello Ada, 2+3=5


## 21. Reversing and repeating strings
- Repeat: no builtin. Use a loop, `printf '%*s' N ''` plus `tr` to fill, or `printf` format reuse.
- Reverse: `rev` exists on macOS and Linux but is not POSIX. A pure-bash loop works everywhere.

In [32]:
%%bash
s="stressed"
rev_loop() { local in=$1 out= i; for ((i = ${#in} - 1; i >= 0; i--)); do out+=${in:i:1}; done; printf '%s\n' "$out"; }
rev_loop "$s"
command -v rev >/dev/null && echo "rev:  $(rev <<< "$s")"
echo "awk:  $(echo "$s" | awk '{ for (i = length($0); i > 0; i--) printf "%s", substr($0, i, 1); print "" }')"
# repeating
repeat() { local n=$2 out=; while (( n-- > 0 )); do out+=$1; done; printf '%s\n' "$out"; }
repeat ab 3
printf '%*s\n' 10 '' | tr ' ' '-'                 # a line of 10 dashes
printf -- '-%.0s' 1 2 3 4 5; echo                  # format reuse: %.0s prints nothing, the dash prints per argument
# palindrome check using the reverse helper
is_pal() { [ "$1" = "$(rev_loop "$1")" ]; }
for w in level hello; do if is_pal "$w"; then echo "$w: palindrome"; else echo "$w: no"; fi; done

desserts
rev:  desserts
awk:  desserts
ababab
----------
-----
level: palindrome
hello: no


## 22. Defaults on array elements and the three different `#` counts
- The `:-`, `:=`, `:+`, `:?` family also works on array elements (see `01-Basics/01-Fundamentals` for the operators).
- `${a[5]:-none}` provides a fallback for a missing element; with the colon an empty element also counts as missing.
- `${#var}` counts characters, `${#a[@]}` counts elements, `${#a[0]}` counts characters of element 0: do not mix them up.
- `${var@Q}`/`${var@U}` transforms need bash 4.4/5.1, not available here.

In [33]:
%%bash
a=(x "" z)
echo "[${a[0]:-none}] [${a[1]:-none}] [${a[1]-none}] [${a[5]:-none}]"
v="hello"
echo "chars: ${#v}   elements of v read as an array: ${#v[@]}   (a plain variable counts as a 1-element array)"
list=(apple banana)
echo "chars in element 0: ${#list[0]}   elements: ${#list[@]}"

[x] [none] [] [none]
chars: 5   elements of v read as an array: 1   (a plain variable counts as a 1-element array)
chars in element 0: 5   elements: 2
